In [11]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DecimalType
from src.spark_session import get_spark
from src.config import PAYMENTS_RAW_PATH, PAYMENTS_SILVER_PATH

In [2]:
spark = get_spark("SilverPayments")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/06 13:33:02 WARN Utils: Your hostname, Branimirs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.199 instead (on interface en0)
26/08/06 13:33:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/branimiranastasov/PycharmProjects/azure_retail_lakehouse/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/06 13:33:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applic

In [3]:
payments_raw_df = spark.read.option("header", True).option("inferSchema", False).csv(str(PAYMENTS_RAW_PATH))
payments_raw_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: string (nullable = true)
 |-- payment_value: string (nullable = true)



In [4]:
payments_raw_df.show(5, truncate=False)

+--------------------------------+------------------+------------+--------------------+-------------+
|order_id                        |payment_sequential|payment_type|payment_installments|payment_value|
+--------------------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b1e8b2acac839d17|1                 |credit_card |8                   |99.33        |
|a9810da82917af2d9aefd1278f1dcfa0|1                 |credit_card |1                   |24.39        |
|25e8ea4e93396b6fa0d3dd708e76c1bd|1                 |credit_card |1                   |65.71        |
|ba78997921bbcdc1373bb41e913ab953|1                 |credit_card |8                   |107.78       |
|42fdf880ba16b47b59251dd489d4441a|1                 |credit_card |2                   |128.45       |
+--------------------------------+------------------+------------+--------------------+-------------+
only showing top 5 rows


In [6]:
order_id_counts = (
    payments_raw_df
    .select("order_id")
    .distinct()
    .count()
)

print("Rows: ", payments_raw_df.count())
print("Columns: ", payments_raw_df.columns)
print("Distinct order_ids: ", order_id_counts)

Rows:  103886
Columns:  ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
Distinct order_ids:  99440


In [7]:
null_order_id = (
    payments_raw_df
    .filter(F.col("order_id").isNull())
)

null_order_id.show()

+--------+------------------+------------+--------------------+-------------+
|order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------+------------------+------------+--------------------+-------------+
+--------+------------------+------------+--------------------+-------------+



In [8]:
order_id_multiple_rows = (
    payments_raw_df
    .groupBy("order_id")
    .agg(
        F.count("*").alias("order_rows_count")
    )
    .orderBy(F.col("order_rows_count").desc())
)

order_id_multiple_rows.show(20, truncate=False)

+--------------------------------+----------------+
|order_id                        |order_rows_count|
+--------------------------------+----------------+
|fa65dad1b0e818e3ccc5cb0e39231352|29              |
|ccf804e764ed5650cd8759557269dc13|26              |
|285c2e15bebd4ac83635ccc563dc71f4|22              |
|895ab968e7bb0d5659d16cd74cd1650c|21              |
|fedcd9f7ccdc8cba3a18defedd1a5547|19              |
|ee9ca989fc93ba09a6eddc250ce01742|19              |
|21577126c19bf11a0b91592e5844ba78|15              |
|4bfcba9e084f46c8e3cb49b0fa6e6159|15              |
|3c58bffb70dcf45f12bdf66a3c215905|14              |
|4689b1816de42507a7d63a4617383c59|14              |
|cf101c3abd3c061ca9f78c1bbb1125af|13              |
|73df5d6adbeea12c8ae03df93f346e86|13              |
|4fb76fa13b108a0d0478483421b0992c|13              |
|465c2e1bee4561cb39e0db8c5993aafc|12              |
|1a611328643ae11146ba09a4425d2e12|12              |
|67d83bd36ec2c7fb557742fb58837659|12              |
|1d9a9731b9c

In [9]:
payments_raw_df.filter(F.col("order_id") == "6d58638e32674bebee793a47ac4cbadc").show()

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|6d58638e32674bebe...|                 7|     voucher|                   1|        10.45|
|6d58638e32674bebe...|                12|     voucher|                   1|        13.53|
|6d58638e32674bebe...|                 1|     voucher|                   1|        12.53|
|6d58638e32674bebe...|                 4|     voucher|                   1|         2.51|
|6d58638e32674bebe...|                 5|     voucher|                   1|         2.24|
|6d58638e32674bebe...|                 9|     voucher|                   1|        10.45|
|6d58638e32674bebe...|                 6|     voucher|                   1|        15.67|
|6d58638e32674bebe...|                 3|     voucher|                   1|         2.87|
|6d58638e3

In [10]:
distinct_payment_keys = (
    payments_raw_df
    .select("order_id", "payment_sequential")
    .distinct()
    .count()
)

print("Rows: ", payments_raw_df.count())
print("Distinct payment keys: ", distinct_payment_keys)
print("Composite key is unique: ", distinct_payment_keys == payments_raw_df.count())

Rows:  103886
Distinct payment keys:  103886
Composite key is unique:  True


In [12]:
payments_schema = StructType([
    StructField("order_id", StringType(), nullable=False),
    StructField("payment_sequential", IntegerType(), nullable=False),
    StructField("payment_type", StringType(), nullable=True),
    StructField("payment_installments", IntegerType(), nullable=True),
    StructField("payment_value", DecimalType(10, 2), nullable=True),
])

In [13]:
payments_typed_df = spark.read.option("header", True).schema(payments_schema).csv(str(PAYMENTS_RAW_PATH))
payments_typed_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: decimal(10,2) (nullable = true)



In [14]:
payment_sequential_null_counts = payments_typed_df.filter(F.col("payment_sequential").isNull())

print("Payment sequential null values: ", payment_sequential_null_counts.count())

Payment sequential null values:  0


In [15]:
payments_cleand_df = (
    payments_typed_df
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("payment_sequential").isNotNull())
    .withColumn(
        "payment_type",
        F.lower(F.trim(F.col("payment_type"))),
    )
    .dropDuplicates(["order_id", "payment_sequential"])
)

payments_cleand_df.show(10, truncate=False)

+--------------------------------+------------------+------------+--------------------+-------------+
|order_id                        |payment_sequential|payment_type|payment_installments|payment_value|
+--------------------------------+------------------+------------+--------------------+-------------+
|00018f77f2f0320c557190d7a144bdd3|1                 |credit_card |3                   |259.83       |
|00061f2a7bc09da83e415a52dc8a4af1|1                 |credit_card |3                   |68.87        |
|0006ec9db01a64e59a68b2c340bf65a7|1                 |credit_card |4                   |97.32        |
|00130c0eee84a3d909e75bc08c5c3ca1|1                 |boleto      |1                   |35.84        |
|0014ae671de39511f7575066200733b7|1                 |boleto      |1                   |30.60        |
|0015ebb40fb17286bea51d4607c4733c|1                 |credit_card |1                   |37.00        |
|001ab0a7578dd66cd4b0a71f5b6e1e41|1                 |boleto      |1               

In [18]:
payments_cleand_df.coalesce(2).write.mode("overwrite").parquet(str(PAYMENTS_SILVER_PATH))